**Nome:** Valdir de Souza Carvalho Neto

**Turma:** SI 241

In [ ]:
!pip install gurobipy

import gurobipy as gp
from gurobipy import GRB
import numpy as np
print(f"Gurobi: {gp.gurobi.version()}")

### MODELAGEM MATEMÁTICA

**Função Objetivo (Minimizar custos de processamento e transporte):**
$$
\begin{aligned}
\min Z &= 6 N_{\text{Std,R1}} + 8 N_{\text{Std,R2}} + 9 N_{\text{Std,R3}} + 9 N_{\text{Prem,R1}} + 11 N_{\text{Prem,R2}} + 12 N_{\text{Prem,R3}} \\
       &\quad + 10 S_{\text{Std,R1}} + 8 S_{\text{Std,R2}} + 7 S_{\text{Std,R3}} + 11 S_{\text{Prem,R1}} + 9 S_{\text{Prem,R2}} + 8 S_{\text{Prem,R3}}
\end{aligned}
$$

**Sujeito a:**

*Capacidade de CPU (Norte e Sul, respectivamente):*
$$
\begin{aligned}
2 N_{\text{Std,R1}} + 2 N_{\text{Std,R2}} + 2 N_{\text{Std,R3}} + 3 N_{\text{Prem,R1}} + 3 N_{\text{Prem,R2}} + 3 N_{\text{Prem,R3}} &\leq 80 \\
2 S_{\text{Std,R1}} + 2 S_{\text{Std,R2}} + 2 S_{\text{Std,R3}} + 3 S_{\text{Prem,R1}} + 3 S_{\text{Prem,R2}} + 3 S_{\text{Prem,R3}} &\leq 70
\end{aligned}
$$

*Capacidade de Armazenamento (Norte e Sul, respectivamente):*
$$
\begin{aligned}
N_{\text{Std,R1}} + N_{\text{Std,R2}} + N_{\text{Std,R3}} + 2 N_{\text{Prem,R1}} + 2 N_{\text{Prem,R2}} + 2 N_{\text{Prem,R3}} &\leq 50 \\
S_{\text{Std,R1}} + S_{\text{Std,R2}} + S_{\text{Std,R3}} + 2 S_{\text{Prem,R1}} + 2 S_{\text{Prem,R2}} + 2 S_{\text{Prem,R3}} &\leq 45
\end{aligned}
$$

*Atendimento Exato das Demandas:*
$$
\begin{aligned}
N_{\text{Std,R1}} + S_{\text{Std,R1}} &= 10 \\
N_{\text{Std,R2}} + S_{\text{Std,R2}} &= 12 \\
N_{\text{Std,R3}} + S_{\text{Std,R3}} &= 8 \\
N_{\text{Prem,R1}} + S_{\text{Prem,R1}} &= 6 \\
N_{\text{Prem,R2}} + S_{\text{Prem,R2}} &= 8 \\
N_{\text{Prem,R3}} + S_{\text{Prem,R3}} &= 6
\end{aligned}
$$

*Domínio das Variáveis (Não-negatividade):*
$$
N_{j,k}, S_{j,k} \geq 0 \quad \text{para todos os serviços e regiões}
$$

In [3]:
datacenters = ["Norte", "Sul"]
servicos = ["Standard", "Premium"]
regioes = ["Regiao1", "Regiao2", "Regiao3"]
custos = {
    ("Norte", "Standard", "Regiao1"): 6,  # 4 de proc. + 2 de frete
    ("Norte", "Standard", "Regiao2"): 8,  # 4 de proc. + 4 de frete
    ("Norte", "Standard", "Regiao3"): 9,  # 4 de proc. + 5 de frete
    ("Sul", "Standard", "Regiao1"): 10,   # 5 de proc. + 5 de frete
    ("Sul", "Standard", "Regiao2"): 8,    # 5 de proc. + 3 de frete
    ("Sul", "Standard", "Regiao3"): 7,    # 5 de proc. + 2 de frete

    ("Norte", "Premium", "Regiao1"): 9,   # 7 de proc. + 2 de frete
    ("Norte", "Premium", "Regiao2"): 11,  # 7 de proc. + 4 de frete
    ("Norte", "Premium", "Regiao3"): 12,  # 7 de proc. + 5 de frete
    ("Sul", "Premium", "Regiao1"): 11,    # 6 de proc. + 5 de frete
    ("Sul", "Premium", "Regiao2"): 9,     # 6 de proc. + 3 de frete
    ("Sul", "Premium", "Regiao3"): 8,     # 6 de proc. + 2 de frete
}
capacidade_cpu = {"Norte": 80, "Sul": 70}
capacidade_armaz = {"Norte": 50, "Sul": 45}
consumo_cpu = {"Standard": 2, "Premium": 3}
consumo_armaz = {"Standard": 1, "Premium": 2}
demanda = {
    ("Standard", "Regiao1"): 10,
    ("Standard", "Regiao2"): 12,
    ("Standard", "Regiao3"): 8,
    ("Premium", "Regiao1"): 6,
    ("Premium", "Regiao2"): 8,
    ("Premium", "Regiao3"): 6
}
m = gp.Model("modelo")
m.Params.OutputFlag = 0
m.Params.Method = 0

x = m.addVars(datacenters, servicos, regioes, lb=0, name="envio")
m.setObjective(
    gp.quicksum(custos[i, j, k] * x[i, j, k] for i in datacenters for j in servicos for k in regioes),
    GRB.MINIMIZE
    )

for j in servicos:
  for k in regioes:
    m.addConstr(
        gp.quicksum(x[i, j, k] for i in datacenters) == demanda[j, k],
        name=f"demanda_{j}_{k}"
    )

for i in datacenters:
  m.addConstr(
      gp.quicksum(consumo_cpu[j] * x[i, j, k] for j in servicos for k in regioes) <= capacidade_cpu[i],
      name=f"cap_cpu_{i}"
  )

for i in datacenters:
  m.addConstr(
        gp.quicksum(consumo_armaz[j] * x[i, j, k] for j in servicos for k in regioes) <= capacidade_armaz[i],
        name=f"cap_armaz_{i}"
  )

m.optimize()

if m.Status == GRB.OPTIMAL:
  print("=-= Plano Ótimo de Processamento e Distribuição =-=")
  for i, j, k in x.keys():
    valor = x[i, j, k].X
    if valor > 1e-6:
      print(f"- Processar {valor:.0f} lote(s) de '{j}' no Datacenter {i} e enviar para a {k}.")

  print(f"\nCusto mínimo total: R$ {m.ObjVal:.2f}")
else:
  print(f"O modelo não encontrou solução ótima. Status: {m.Status}")

Restricted license - for non-production use only - expires 2027-11-29
=-= Plano Ótimo de Processamento e Distribuição =-=
- Processar 10 lote(s) de 'Standard' no Datacenter Norte e enviar para a Regiao1.
- Processar 12 lote(s) de 'Standard' no Datacenter Norte e enviar para a Regiao2.
- Processar 6 lote(s) de 'Premium' no Datacenter Norte e enviar para a Regiao1.
- Processar 8 lote(s) de 'Standard' no Datacenter Sul e enviar para a Regiao3.
- Processar 8 lote(s) de 'Premium' no Datacenter Sul e enviar para a Regiao2.
- Processar 6 lote(s) de 'Premium' no Datacenter Sul e enviar para a Regiao3.

Custo mínimo total: R$ 386.00
